<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.1-prompting-fundamentals/notebooks/exercises-3_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.1: Prompting Fundamentals in Python

*Module 3 · Prompt Engineering & Context Design*

> Same model. Same API. A good prompt gets a 5/10 answer. A great prompt gets a 9/10. This lesson teaches you the difference — tested with real API calls so you see the proof.

---

## About this notebook

This notebook contains the **15 runnable code examples** from the published lesson `lesson-3.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Setup: The Helper Function We'll Use Throughout — `setup.py`
2. Step 1 — Prompt Anatomy — The 4 Parts of Every Prompt — `prompt_anatomy.py`
3. Step 2 — Instruction Clarity — Be Specific or Get Garbage — `part1_vague_vs_specific.py`
4. Step 2 — Instruction Clarity — Be Specific or Get Garbage — `part1_all_rules.py`
5. Step 3 — Zero-Shot vs Few-Shot — The Power of Examples — `few_shot.py`
6. Step 4 — Role Assignment — Make the AI an Expert — `part2_role_comparison.py`
7. Step 4 — Role Assignment — Make the AI an Expert — `part2_system_prompt.py`
8. Step 5 — Delimiters & XML Tags — Separate Data from Instructions — `delimiters.py`
9. Step 6 — Output Formatting — Get JSON, Tables, and Structured Data — `part3_json_output.py`
10. Step 6 — Output Formatting — Get JSON, Tables, and Structured Data — `part3_multiple_formats.py`
11. Step 7 — Chain-of-Thought Basics — "Think Step by Step" — `cot_basics.py`
12. Step 8 — Temperature & Parameters — Control the Randomness — `parameters.py`
13. Step 9 — Failure Modes — Hallucination, Sycophancy & Grounding — `failure_modes.py`
14. Step 10 — Prompt Iteration & Testing — Prove Your Prompt Works — `part4_prompt_tester.py`
15. Step 11 — Project: Product Review Analyzer — `project_review_analyzer.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai requests pydantic


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


#### Setup: The Helper Function We'll Use Throughout

**`setup.py`** · _Block 1 of 15_

SETUP: One function to call the AI — We'll use this in every example below.


In [ ]:
# =============================================
# SETUP: One function to call the AI
# We'll use this in every example below.
# =============================================

# pip install google-generativeai
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def ask(prompt: str, temperature: float = 0.3) -> str:
    """Send a prompt to Gemini and get a response."""
    model = genai.GenerativeModel(
        "gemini-2.0-flash",
        generation_config={"temperature": temperature}
    )
    response = model.generate_content(prompt)
    return response.text

# Quick test
print(ask("Say hello in one word."))
# → "Hello!"


### Step 1 · Prompt Anatomy — The 4 Parts of Every Prompt

Every effective prompt has the same structure, whether you use Gemini, Claude, or GPT.

**`prompt_anatomy.py`** · _Block 2 of 15_

THE 4 PARTS OF EVERY PROMPT — Part 1: INSTRUCTION — what to do


In [ ]:
# THE 4 PARTS OF EVERY PROMPT
# Part 1: INSTRUCTION — what to do
# Part 2: CONTEXT    — background info
# Part 3: INPUT DATA — the specific thing to process
# Part 4: OUTPUT FORMAT — how to structure the response

# BAD: Only instruction
bad = "Tell me about Hyderabad"

# GOOD: All 4 parts
good = """[INSTRUCTION] Write a travel guide paragraph.
[CONTEXT] For a first-time US visitor, 3-day trip.
[INPUT] City: Hyderabad, India.
[OUTPUT] Top 3 attractions, best food, one insider tip.
         Keep under 150 words."""

print("Prompt Anatomy:")
print("  INSTRUCTION + CONTEXT + INPUT + OUTPUT FORMAT")
print("  Missing any part → vague output")
print("  All 4 present → reliable, useful output")
print("  Anthropic golden rule: if a colleague couldn't follow")
print("  your prompt with zero context, the AI won't either.")


> **What just happened?** Every prompt has 4 parts: instruction , context , input , output format . The bad prompt had only instruction. The good prompt had all 4 and got a dramatically better result. Anthropic's golden rule: if a colleague couldn't follow your prompt with zero context, the AI won't either.


### Step 2 · Instruction Clarity — Be Specific or Get Garbage

The #1 prompting mistake: being vague. Let's see the difference.

**`part1_vague_vs_specific.py`** · _Block 3 of 15_

EXPERIMENT 1: Vague vs Specific Instructions — Same task. Different prompt clarity.


In [ ]:
# =============================================
# EXPERIMENT 1: Vague vs Specific Instructions
# Same task. Different prompt clarity.
# Watch how the output quality changes.
# =============================================

# ❌ VAGUE PROMPT
vague = "Tell me about machine learning."

# ✅ SPECIFIC PROMPT
specific = """Explain machine learning to a 15-year-old student 
who has never heard of it. Use exactly 3 sentences. 
Include one real-world example they'd relate to (like 
Instagram or YouTube). Don't use any technical jargon."""

print("═══ VAGUE PROMPT ═══")
print(ask(vague))
print()

print("═══ SPECIFIC PROMPT ═══")
print(ask(specific))

# The vague prompt gives a long, generic, textbook-like response.
# The specific prompt gives a concise, relatable, age-appropriate answer.
# Same model. Same API. The ONLY difference is the prompt.


> **What just happened?** The vague prompt produced a generic, 200+ word essay. The specific prompt produced exactly 3 sentences with an Instagram example and no jargon. We added 4 constraints: audience (15-year-old), length (3 sentences), content (real-world example), and style (no jargon). Each constraint narrows the output toward exactly what we need.


#### The 5 Rules of Clear Instructions

**`part1_all_rules.py`** · _Block 4 of 15_

ALL 5 RULES IN ONE PROMPT — Watch how combining constraints produces


In [ ]:
# =============================================
# ALL 5 RULES IN ONE PROMPT
# Watch how combining constraints produces
# exactly the output you need.
# =============================================

prompt = """
You are writing for a tech blog. Your audience is beginner 
programmers who know Python but not AI.               [Rule 1: audience]

Explain what a "neural network" is in exactly 4 sentences.  [Rule 2: length]

Do NOT use math equations or Greek letters.            [Rule 3: what NOT to do]

Format each sentence like this:                        [Rule 4: example format]
  1. [Simple analogy]
  2. [How it works in plain words]
  3. [One real-world use case]
  4. [Why a Python developer should care]

Think step by step before writing.                     [Rule 5: break into steps]
"""

result = ask(prompt)
print(result)

# The output will be:
# - Exactly 4 numbered sentences
# - No math or Greek letters
# - Written for Python developers
# - Following the exact format we specified


### Step 3 · Zero-Shot vs Few-Shot — The Power of Examples

The single most universally taught prompting technique. Every top course covers this.

**`few_shot.py`** · _Block 5 of 15_

ZERO-SHOT: No examples


In [ ]:
# ZERO-SHOT: No examples
zero = "Classify: 'The food was terrible' → positive/negative?"

# FEW-SHOT: 3 examples teach the pattern
few = """Classify reviews:
Review: "Amazing service!" → positive
Review: "Worst ever" → negative
Review: "It was okay" → neutral

Review: "Terrible food but nice ambiance" →"""

print("""WHEN TO USE:
  Zero-shot:  Simple tasks (translation, summary)
  Few-shot:   Classification, extraction, custom formats (2-5 examples)
  Many-shot:  100K+ context, 50-100 examples for hard tasks
  
  KEY: Format of examples matters more than correct labels!
  Few-shot teaches PATTERN, not knowledge.
  Rule: if zero-shot >90% accuracy, don't add examples.""")


> **What just happened?** Zero-shot = no examples. Few-shot = 2-5 examples teach the pattern. Research (Min et al. 2022): example format matters more than correct labels . Rule: zero-shot for simple tasks, few-shot for classification/extraction. Deep dive in Lesson 3.2.


### Step 4 · Role Assignment — Make the AI an Expert

"You are a senior doctor" produces very different output than "You are a comedian."

**`part2_role_comparison.py`** · _Block 6 of 15_

EXPERIMENT 2: Same question, different roles — Watch the answer COMPLETELY change based on


In [ ]:
# =============================================
# EXPERIMENT 2: Same question, different roles
# Watch the answer COMPLETELY change based on
# who the AI thinks it is.
# =============================================

question = "How should I handle errors in my Python API?"

roles = {
    "No role": question,
    
    "Junior intern": f"""You are a junior intern who just started learning Python 
last month. Someone asks you: {question}
Answer in your own words as this character.""",
    
    "Senior engineer": f"""You are a senior backend engineer at Google with 12 years 
of experience building production APIs that handle millions of 
requests per day. Someone asks you: {question}
Give your expert, production-grade answer.""",
    
    "Stand-up comedian": f"""You are a stand-up comedian who also happens to know Python. 
Someone asks you: {question}
Answer accurately but make it funny. Include at least 2 jokes.""",
}

for role_name, prompt in roles.items():
    print(f"\n{'═' * 50}")
    print(f"ROLE: {role_name}")
    print(f"{'═' * 50}")
    print(ask(prompt))

# You'll see:
# No role → generic, textbook answer
# Junior intern → simple, unsure, basic try/except
# Senior engineer → production patterns, logging, monitoring, retry
# Comedian → accurate but hilarious ("try/except is like a seatbelt...")


> **What just happened?** Same question, 4 completely different answers. The "senior engineer" role gives production-grade advice (logging, monitoring, custom exception classes). The "no role" version gives a generic try/except tutorial. Roles change the depth, tone, vocabulary, and thinking pattern of the response. This is one of the most powerful prompting techniques.


#### System Prompts: The Professional Way to Set Roles

**`part2_system_prompt.py`** · _Block 7 of 15_

SYSTEM PROMPTS: Set the role ONCE, — it applies to every message after.


In [ ]:
# =============================================
# SYSTEM PROMPTS: Set the role ONCE,
# it applies to every message after.
# This is how ChatGPT custom instructions work.
# =============================================

import google.generativeai as genai

# Create a model with a system prompt (the role)
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    system_instruction="""You are Dr. Maya, a senior AI researcher 
at a top Indian tech company. You explain complex AI concepts 
using everyday Indian analogies (cricket, Bollywood, chai, 
Indian festivals). You're friendly, encouraging, and always 
relate concepts back to practical Python code.

Rules:
- Always start with a one-line analogy
- Keep answers under 150 words
- End with a one-line Python code hint
- Use "beta/beti" casually when encouraging the student"""
)

# Now every question gets answered "as Dr. Maya"
chat = model.start_chat()

questions = [
    "What is overfitting?",
    "How does attention work in Transformers?",
    "What's the difference between CNN and RNN?",
]

for q in questions:
    print(f"\nStudent: {q}")
    response = chat.send_message(q)
    print(f"Dr. Maya: {response.text}")
    print(f"{'─' * 50}")

# Dr. Maya will explain overfitting like "ordering only butter chicken
# at every restaurant — you memorize one dish but can't handle anything new"
# The system prompt personality stays consistent across ALL messages!


> **What just happened?** We created a persistent AI character — "Dr. Maya" — using a system prompt. Every answer uses Indian analogies, stays under 150 words, includes a Python hint, and maintains a warm, encouraging tone. The system prompt is set once and applies to all messages. This is how you build customer support bots, tutoring systems, and AI assistants with consistent personalities.


### Step 5 · Delimiters & XML Tags — Separate Data from Instructions

The #1 defense against prompt injection. Essential for production prompts.

**`delimiters.py`** · _Block 8 of 15_

DANGEROUS: User input mixed with instructions


In [ ]:
# DANGEROUS: User input mixed with instructions
bad = f"Summarize: {user_input}"  # prompt injection risk!

# SAFE: Claude — XML tags (fine-tuned for this)
claude = """Summarize the document.
<document>{user_input}</document>
Output 3 sentences."""

# SAFE: GPT — Markdown headers
gpt = """### Instructions
Summarize in 3 sentences.
### Document
```{user_input}```"""

print("""DELIMITER CHEAT SHEET:
  Claude:  XML tags (<document>, <instructions>) — fine-tuned!
  GPT:     Markdown headers (###) or triple backticks
  Gemini:  Both work, XML preferred
  
  WHY: Prevents injection, improves accuracy, enables extraction
  Deep dive → Lesson 3.7 (Adversarial Defenses)""")


> **What just happened?** Delimiters separate instructions from data. Claude uses XML tags (fine-tuned), GPT uses markdown . In production, EVERY user input must be wrapped in delimiters to prevent prompt injection. Also covered in Lesson 3.7.


### Step 6 · Output Formatting — Get JSON, Tables, and Structured Data

AI outputs are useless to your app if they're free-form text. Let's force structure.

#### 3a: Getting JSON Output

**`part3_json_output.py`** · _Block 9 of 15_

GET CLEAN JSON FROM THE AI — The trick: show the exact shape you want.


In [ ]:
# =============================================
# GET CLEAN JSON FROM THE AI
# The trick: show the exact shape you want.
# =============================================

import json

review = """Just bought the Sony WH-1000XM5 headphones for ₹29,999 
from Amazon. The noise cancelling is absolutely incredible — I can't 
hear anything on the metro now. Battery lasts forever. Only complaint 
is the touch controls are a bit fiddly. 4 out of 5 stars from me!"""

prompt = f"""Analyze this product review and extract structured information.

Review: "{review}"

Return ONLY valid JSON with this exact structure (no extra text, no markdown):
{{
    "product_name": "string",
    "brand": "string",
    "price": number or null,
    "currency": "string",
    "rating": number (1-5),
    "sentiment": "positive" | "negative" | "mixed",
    "pros": ["string", "string"],
    "cons": ["string", "string"],
    "purchase_platform": "string or null"
}}"""

print("Asking AI to extract structured data...\n")
result = ask(prompt)
print("Raw AI output:")
print(result)

# Parse the JSON
try:
    data = json.loads(result)
    print(f"\n✅ Successfully parsed! Here's the structured data:")
    print(f"   Product: {data['product_name']}")
    print(f"   Price:   {data['currency']}{data['price']:,}")
    print(f"   Rating:  {data['rating']}/5")
    print(f"   Pros:    {', '.join(data['pros'])}")
    print(f"   Cons:    {', '.join(data['cons'])}")
except json.JSONDecodeError as e:
    print(f"❌ Failed to parse JSON: {e}")
    print("   Tip: Add 'Return ONLY valid JSON, no markdown' to prompt")


> **What just happened?** We gave the AI a review in free text and asked for structured JSON back. The key: we showed the exact JSON shape we wanted — field names, types, even possible values for enums like "positive | negative | mixed." The AI returned parseable JSON that our Python code could use directly. This pattern — free text in, structured JSON out — is the foundation of every production AI application.


#### 3b: Getting Multiple Formats

**`part3_multiple_formats.py`** · _Block 10 of 15_

FORMAT CONTROL: Get ANY structure you want — Markdown table, bullet list, CSV, YAML, XML —


In [ ]:
# =============================================
# FORMAT CONTROL: Get ANY structure you want
# Markdown table, bullet list, CSV, YAML, XML —
# just show the format in the prompt.
# =============================================

topic = "Compare Python, JavaScript, and Rust for backend development."

formats = {
    "Markdown table": f"""{topic}
Format as a markdown table with columns: Language | Speed | Ease of Learning | Best For | Popular Framework
Include exactly 3 rows (one per language).""",

    "Bullet list": f"""{topic}
Format as a bullet list. For each language, use this format:
- **Language name**: One sentence summary. Best for: [use case]. Learning curve: [easy/medium/hard].""",

    "CSV": f"""{topic}
Return ONLY a CSV with headers: language,speed_rating,ease_rating,best_for,framework
Use numbers 1-10 for ratings. No extra text.""",

    "Decision flowchart": f"""{topic}
Format as a decision tree using text:
→ If [condition] → Use [language] because [reason]
→ If [condition] → Use [language] because [reason]
→ Otherwise → Use [language] because [reason]""",
}

for format_name, prompt in formats.items():
    print(f"\n{'═' * 50}")
    print(f"FORMAT: {format_name}")
    print(f"{'═' * 50}")
    print(ask(prompt))

# Same content, 4 completely different formats.
# The AI adapts its structure to whatever you ask for.


### Step 7 · Chain-of-Thought Basics — "Think Step by Step"

Three words that improve math accuracy by 40%+.

**`cot_basics.py`** · _Block 11 of 15_

DIRECT — model often gets math wrong


In [ ]:
# DIRECT — model often gets math wrong
direct = "Shirt ₹800, 25% off, ₹100 coupon. Final price?"

# ZERO-SHOT CoT — just add the magic phrase
cot = """Shirt ₹800, 25% off, ₹100 coupon. Final price?
Think step by step before answering."""

# STRUCTURED CoT — production pattern
structured = """Shirt ₹800, 25% off, ₹100 coupon.
Reason inside <thinking> tags. Final answer in <answer> tags.

<thinking>[reasoning]</thinking>
<answer>[price only]</answer>"""

print("""THREE CoT TIERS:
  1. Zero-shot: "Think step by step" (+20-40% on math)
  2. Guided:    Specify reasoning steps explicitly
  3. Structured: <thinking> + <answer> tags (production)
  
  USE for: math, logic, multi-step analysis
  SKIP for: simple classification, translation
  Deep dive → Lesson 3.3""")


> **What just happened?** "Think step by step" (Kojima et al. 2022) improves math/logic by 20-40%. Three tiers: zero-shot CoT, guided CoT, structured CoT with tags. Production tip: parse <answer> tags programmatically, keep <thinking> for debugging. Deep dive in Lesson 3.3.


### Step 8 · Temperature & Parameters — Control the Randomness

Same prompt, different temperature → completely different outputs.

**`parameters.py`** · _Block 12 of 15_


In [ ]:
import google.generativeai as genai
model = genai.GenerativeModel("gemini-2.0-flash")
prompt = "Write a tagline for a Hyderabad biryani restaurant"

# Temperature 0.0 — deterministic (facts, classification)
r1 = model.generate_content(prompt, generation_config={"temperature": 0.0})
# Temperature 0.7 — balanced (default for most tasks)
r2 = model.generate_content(prompt, generation_config={"temperature": 0.7})
# Temperature 1.5 — very creative (may produce nonsense)
r3 = model.generate_content(prompt, generation_config={"temperature": 1.5})

print("""SETTINGS BY TASK:
  Classification: temperature=0.0    Creative: 0.7-1.0
  Summarization:  0.3-0.5            Code: 0.0-0.2
  
  max_tokens includes BOTH input + output!
  top_p: 0.95 default, lower = more focused""")


> **What just happened?** Temperature controls randomness: 0.0 = deterministic, 1.0+ = creative. Use 0.0 for facts/classification, 0.7 for creative tasks. Critical: max_tokens includes input AND output — budget accordingly.


### Step 9 · Failure Modes — Hallucination, Sycophancy & Grounding

The AI will confidently make things up. Here is how to prevent it.

**`failure_modes.py`** · _Block 13 of 15_


In [ ]:
print("""FAILURE MODES:
1. HALLUCINATION — invents false facts confidently
   FIX: "Answer ONLY based on the document below.
        If not found, say 'Not in provided text.'"

2. SYCOPHANCY — agrees even when you are wrong
   FIX: "First analyze independently, then compare."

3. INSTRUCTION FAILURES — ignores constraints
   FIX: Put constraints at END (recency bias)
        + Use structured outputs (Lesson 3.5)

4. INDIAN DOMAIN GAPS — 58% on IndicParam benchmark
   FIX: Always provide reference text for Indian law,
        GST, SEBI. Few-shot with Indian examples.

DEFAULT ASSUMPTION: The model WILL hallucinate
unless you ground it with reference text.""")


> **What just happened?** Four failure modes: Hallucination (ground with reference text), Sycophancy (ask for independent analysis), Instruction failures (put constraints at end), Indian domain gaps (58% accuracy — always provide reference text). Default: assume hallucination unless grounded.


### Step 10 · Prompt Iteration & Testing — Prove Your Prompt Works

Don't just try a prompt once and hope. Test it 10 times and measure reliability.

**`part4_prompt_tester.py`** · _Block 14 of 15_

PROMPT TESTING FRAMEWORK — Test a prompt 10 times and measure:


In [ ]:
# =============================================
# PROMPT TESTING FRAMEWORK
# Test a prompt 10 times and measure:
# - Does it return valid JSON?
# - Does it include required fields?
# - Is the sentiment correct?
# =============================================

import json

def test_prompt(prompt, checks, num_runs=10):
    """Run a prompt multiple times and measure success rate."""
    results = {check_name: 0 for check_name in checks}
    
    print(f"Running {num_runs} tests...\n")
    
    for i in range(num_runs):
        response = ask(prompt)
        
        for check_name, check_fn in checks.items():
            try:
                if check_fn(response):
                    results[check_name] += 1
            except:
                pass  # check failed
        
        status = "✅" if all(
            checks[c](response) for c in checks
        ) else "❌"
        print(f"  Run {i+1:2d}: {status}")
    
    print(f"\nResults ({num_runs} runs):")
    for check_name, passes in results.items():
        rate = passes / num_runs * 100
        emoji = "✅" if rate >= 90 else "⚠️" if rate >= 70 else "❌"
        print(f"  {emoji} {check_name}: {rate:.0f}% ({passes}/{num_runs})")

# ── Define the prompt and checks ──

prompt = """Analyze this review and return ONLY valid JSON:
{"sentiment": "positive" or "negative", "rating": 1-5, "summary": "one sentence"}

Review: "Amazing battery life but the screen is dim. 3 stars."
"""

checks = {
    "Valid JSON":     lambda r: json.loads(r) is not None,
    "Has sentiment":  lambda r: "sentiment" in json.loads(r),
    "Has rating":     lambda r: "rating" in json.loads(r),
    "Rating is 3":    lambda r: json.loads(r).get("rating") == 3,
    "Has summary":    lambda r: len(json.loads(r).get("summary", "")) > 10,
}

test_prompt(prompt, checks, num_runs=10)

# Output: something like
# ✅ Valid JSON: 100% (10/10)
# ✅ Has sentiment: 100% (10/10)
# ✅ Has rating: 100% (10/10)
# ⚠️ Rating is 3: 80% (8/10)    ← sometimes the AI says "mixed" = 2.5
# ✅ Has summary: 100% (10/10)


> **What just happened?** We built a prompt testing framework that runs a prompt 10 times and checks: Is the output valid JSON? Does it have the right fields? Is the rating correct? Is there a summary? This turns prompt engineering from "try and hope" into a measurable, scientific process. In production, you'd run 100+ tests and only deploy prompts with 95%+ success rate.


### Step 11 · Project: Product Review Analyzer

Combine all 3 techniques into one production-ready tool.

**`project_review_analyzer.py`** · _Block 15 of 15_

FINAL PROJECT: Product Review Analyzer — Uses ALL 3 techniques:


In [ ]:
# =============================================
# FINAL PROJECT: Product Review Analyzer
# Uses ALL 3 techniques:
#   ✅ Clear instructions (specific constraints)
#   ✅ Role assignment (system prompt)
#   ✅ Output formatting (JSON with Pydantic)
# =============================================

import google.generativeai as genai
import json
from pydantic import BaseModel, Field
from typing import Optional

# ── Step 1: Define the exact output shape with Pydantic ──
class ReviewAnalysis(BaseModel):
    product: str = Field(description="Product name")
    brand: str = Field(description="Brand name")
    rating: int = Field(ge=1, le=5)
    sentiment: str = Field(description="positive, negative, or mixed")
    pros: list[str] = Field(description="List of positive points")
    cons: list[str] = Field(description="List of negative points")
    price: Optional[float] = None
    recommend: bool = Field(description="Would reviewer recommend?")
    one_line_summary: str = Field(max_length=100)

# ── Step 2: Create the model with a role (system prompt) ──
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    system_instruction="""You are an expert product review analyst for an 
e-commerce platform. You extract structured data from customer reviews 
with high accuracy. You always return valid JSON matching the requested 
schema. You never include markdown formatting or extra text — ONLY the 
JSON object.""",
    generation_config={"temperature": 0.1}  # low temp for consistency
)

# ── Step 3: Build the prompt with clear instructions ──
def analyze_review(review_text: str) -> ReviewAnalysis:
    """Analyze a product review and return structured data."""
    
    prompt = f"""Analyze this product review and extract information.

Review: "{review_text}"

Return ONLY valid JSON matching this exact schema:
{json.dumps(ReviewAnalysis.model_json_schema(), indent=2)}

Rules:
- rating must be 1-5 integer
- sentiment must be exactly "positive", "negative", or "mixed"
- pros and cons must each have at least 1 item
- one_line_summary must be under 100 characters
- recommend should be true if rating >= 3
- Return ONLY the JSON object. No markdown. No extra text."""
    
    response = model.generate_content(prompt)
    
    # Clean and parse
    text = response.text.strip()
    if text.startswith("```"):  # remove markdown fences if present
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    
    # Validate with Pydantic
    return ReviewAnalysis.model_validate_json(text)

# ── Step 4: Test with real reviews ──
reviews = [
    """Bought the OnePlus Buds 3 for ₹3,299. Sound quality is fantastic 
for the price — punchy bass and clear vocals. ANC works well on the 
metro. Battery life is solid at 40+ hours with the case. My only gripe 
is the fit — they fall out when I jog. 4 stars, would recommend for 
non-sporty use!""",
    
    """Terrible experience with this mixer grinder. Motor burned out in 
2 weeks. Customer service was unhelpful. The jars leak. Complete waste 
of ₹2,500. 1 star. Do NOT buy this.""",
    
    """The Kindle Paperwhite is decent. Screen is nice for reading at night. 
But it's slow to load PDFs and the storage fills up fast. For ₹13,999, 
I expected more. 3 stars — it's okay but not great.""",
]

print("Product Review Analyzer")
print("=" * 50)

for i, review in enumerate(reviews):
    print(f"\nReview {i+1}:")
    print(f"  \"{review[:60]}...\"\n")
    
    try:
        analysis = analyze_review(review)
        emoji = {"positive": "👍", "negative": "👎", "mixed": "🤷"}
        print(f"  {emoji.get(analysis.sentiment, '?')} {analysis.product} by {analysis.brand}")
        print(f"  Rating: {'⭐' * analysis.rating} ({analysis.rating}/5)")
        print(f"  Sentiment: {analysis.sentiment}")
        print(f"  Pros: {', '.join(analysis.pros)}")
        print(f"  Cons: {', '.join(analysis.cons)}")
        print(f"  Price: ₹{analysis.price:,.0f}" if analysis.price else "  Price: not mentioned")
        print(f"  Recommend: {'Yes' if analysis.recommend else 'No'}")
        print(f"  Summary: {analysis.one_line_summary}")
    except Exception as e:
        print(f"  ❌ Error: {e}")


> **What just happened?** We built a complete product review analyzer combining all 3 techniques: role (system prompt as expert analyst), clear instructions (specific rules for each field), and output formatting (JSON validated by Pydantic). Free-text reviews go in → clean, validated, structured data comes out. This is a real production pattern used by e-commerce companies to process thousands of reviews automatically.


---

## Wrap-up

You've walked through all 15 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
